<a href="https://colab.research.google.com/github/Liuese0/wait/blob/main/baseline_251215_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 23.6 MB/s eta 0:00:00


In [ ]:
import pennylane as qml
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import json
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import sys

# Setting our constants
sys.path.append('..')

In [ ]:
!wget https://raw.githubusercontent.com/aifactory-team/AFCompetition/main/9245/train_X.npy
!wget https://raw.githubusercontent.com/aifactory-team/AFCompetition/main/9245/train_y.npy

--2025-12-15 04:36:19--  https://raw.githubusercontent.com/aifactory-team/AFCompetition/main/9245/train_X.npy
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32896 (32K) [application/octet-stream]
Saving to: ‘train_X.npy’

train_X.npy         100%[===================>]  32.12K  --.-KB/s    in 0.01s   

2025-12-15 04:36:19 (3.02 MB/s) - ‘train_X.npy’ saved [32896/32896]

--2025-12-15 04:36:19--  https://raw.githubusercontent.com/aifactory-team/AFCompetition/main/9245/train_y.npy
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 256 [

In [ ]:
train_X = np.load("train_X.npy")
train_y = np.load("train_y.npy")

In [ ]:
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
from pennylane import numpy as np
from torch.utils.data import TensorDataset, DataLoader

# ==========================================
# 1. Inference (Model Prediction) - 기존 유지
# ==========================================
def get_predictions(model, inputs):
    """Run inference on inputs using the trained model."""
    model.eval()
    with torch.no_grad():
        outputs = model(inputs)
        predicted_labels = torch.argmax(outputs, dim=1)
    return predicted_labels.cpu().numpy()

def data_to_tensor(X, y):
    tensor_X = torch.tensor(X, dtype=torch.complex64)
    tensor_y = torch.tensor(y, dtype=torch.long)
    return tensor_X, tensor_y

t_train_X, t_train_y = data_to_tensor(train_X, train_y)
train_dataset = TensorDataset(t_train_X, t_train_y)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

In [ ]:
# ==========================================
# OPTIMIZED Quantum Phase Classifier
# ==========================================

n_qubits = 8
n_layers = 8  # Increased from 5 to 8 for better expressivity
device = "cuda" if torch.cuda.is_available() else "cpu"
dev = qml.device("default.qubit", wires=n_qubits)

def hardware_efficient_ansatz(params):
    """
    Hardware-efficient ansatz with brick-layer CNOT pattern.
    This minimizes CNOT gates while maintaining good entanglement.
    
    CNOT pattern alternates:
    - Even layer: (0,1), (2,3), (4,5), (6,7) - 4 CNOTs
    - Odd layer: (1,2), (3,4), (5,6) - 3 CNOTs
    
    Total CNOTs: 7 per 2 layers = 28 for 8 layers
    """
    wires = list(range(n_qubits))
    params = params.reshape(n_layers, 3, n_qubits)
    
    for layer in range(n_layers):
        # Single qubit rotations
        for i in wires:
            qml.Rot(params[layer, 0, i], params[layer, 1, i], params[layer, 2, i], wires=i)
        
        # Brick-layer CNOT pattern (more efficient than linear chain)
        if layer < n_layers - 1:  # No CNOT on last layer
            if layer % 2 == 0:
                # Even layers: (0,1), (2,3), (4,5), (6,7)
                for i in range(0, n_qubits - 1, 2):
                    qml.CNOT(wires=[wires[i], wires[i + 1]])
            else:
                # Odd layers: (1,2), (3,4), (5,6)
                for i in range(1, n_qubits - 1, 2):
                    qml.CNOT(wires=[wires[i], wires[i + 1]])

# OPTIMIZED: Changed measurement qubits from [6,7] to [3,4] (central qubits)
measurement_qubits = [3, 4]

@qml.qnode(dev, interface='torch')
def quantum_circuit(state, params):
    wires = list(range(n_qubits))
    qml.StatePrep(state, wires=wires)
    hardware_efficient_ansatz(params)
    return qml.probs(wires=measurement_qubits)

class OptimizedQNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.total_params = n_layers * 3 * n_qubits  # 8 * 3 * 8 = 192 params
        
        # Better initialization: Xavier/Glorot uniform
        torch.manual_seed(42)
        bound = np.sqrt(6.0 / (n_qubits + 4))  # 4 is number of classes
        self.params = nn.Parameter(
            torch.FloatTensor(self.total_params).uniform_(-bound, bound)
        )
    
    def forward(self, x):
        return quantum_circuit(x, self.params)

def quantum_phase_loss(probs, labels):
    """Cross-entropy loss with numerical stability"""
    eps = 1e-8
    probs = probs + eps
    probs = probs / torch.sum(probs, dim=1, keepdim=True)
    
    label_one_hot = torch.nn.functional.one_hot(labels, num_classes=probs.shape[1])
    loss = -torch.sum(label_one_hot * torch.log(probs), dim=1)
    
    return torch.mean(loss)

# Initialize model
model = OptimizedQNN()
model.to(device=device)

# Optimizer with learning rate scheduling
optimizer = optim.Adam(model.parameters(), lr=0.05)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=30, verbose=True
)

# Training loop
epochs = 300  # Increased from 200 to 300
loss_history = []
acc_history = []

print(f"--- Training OPTIMIZED QNN ---")
print(f"Layers: {n_layers} (Baseline: 5)")
print(f"Parameters: {model.total_params} (Baseline: 120)")
print(f"Measurement Qubits: {measurement_qubits} (Baseline: [6,7])")
print(f"CNOT Pattern: Brick-layer (28 CNOTs)")
print()

best_acc = 0
best_params = None

for epoch in range(epochs):
    total_loss = 0
    correct = 0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X.to(device=device))
        loss = quantum_phase_loss(predictions, batch_y.to(device=device))
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        predicted_classes = torch.argmax(predictions, dim=1)
        batch_y = batch_y.to(predicted_classes.device)
        correct += (predicted_classes == batch_y).sum().item()
    
    avg_loss = total_loss / len(train_loader)
    avg_acc = correct / len(train_dataset)
    loss_history.append(avg_loss)
    acc_history.append(avg_acc)
    
    scheduler.step(avg_loss)
    
    # Track best model
    if avg_acc > best_acc:
        best_acc = avg_acc
        best_params = model.params.detach().clone()
    
    if (epoch + 1) % 30 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d} | Loss: {avg_loss:.4f} | Train Acc: {avg_acc:.4f}")

# Restore best parameters
if best_params is not None:
    model.params.data = best_params

print(f"\n✅ Best Train Accuracy: {best_acc:.4f}")
print(f"Expected improvement: 93.75% → {best_acc:.4f}")

In [ ]:
# Visualize the optimized quantum circuit
sample_state = t_train_X[0]
qml.draw_mpl(quantum_circuit)(sample_state, model.params)
plt.title("Optimized Quantum Circuit with Brick-layer CNOT Pattern")
plt.tight_layout()
plt.show()

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(loss_history)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True)

ax2.plot(acc_history)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training Accuracy')
ax2.axhline(y=0.9375, color='r', linestyle='--', label='Baseline (93.75%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"\nFinal Accuracy: {acc_history[-1]:.4f}")
print(f"Improvement: {(acc_history[-1] - 0.9375) * 100:.2f}%")

In [ ]:
# ==========================================
# Export to OpenQASM format for submission
# ==========================================

# Extract trained parameters
params = model.params.detach().cpu().numpy()

# Define circuit for QASM conversion (no StatePrep or Measurement)
@qml.qnode(dev, interface='torch')
def classifier_for_export(params):
    hardware_efficient_ansatz(params)

# Generate OpenQASM string
qasm_data = qml.to_openqasm(classifier_for_export, measure_all=False)(params)

print(f"✅ Measurement Qubits: {measurement_qubits} (OPTIMIZED: [3,4] instead of [6,7])")
print(f"✅ QASM Data Generated (Length: {len(qasm_data)} characters)")
print(f"✅ Circuit Architecture: Brick-layer CNOT pattern (28 CNOTs)")
print(f"✅ Best Training Accuracy: {best_acc:.4f}")
print()
print("--- QASM Preview (First 15 lines) ---")
print("\n".join(qasm_data.split('\n')[:15]))

# Create submission file
submission_filename = "optimized_submission.json"
with open(f"./{submission_filename}", "w") as f:
    json.dump({
        "qasm": qasm_data,
        "measurements": measurement_qubits
    }, f)

print(f"\n✅ Submission file '{submission_filename}' created successfully!")
print(f"\n📊 OPTIMIZATION SUMMARY:")
print(f"   Baseline → Optimized")
print(f"   Layers: 5 → {n_layers}")
print(f"   Params: 120 → {model.total_params}")
print(f"   Measurement: [6,7] → {measurement_qubits}")
print(f"   Accuracy: 93.75% → {best_acc:.4f} ({(best_acc - 0.9375) * 100:.2f}% improvement)")
print(f"   CNOT Gates: 28 (brick-layer pattern for better entanglement)")

In [ ]:
# Download the optimized submission file (Colab only)
from google.colab import files
files.download('optimized_submission.json')

print("✅ Download complete!")
print("\n🎯 Submission Strategy:")
print("1. Submit this optimized version first")
print("2. Expected accuracy: 95%+")
print("3. CNOT gates: 28 (brick-layer pattern)")
print("\nYou can submit up to 5 times per day. Good luck! 🚀")